# LING 539 — Text Classification Competition

**Task:** Classify text documents into three categories:
- **0** — Not a movie/TV show review
- **1** — Positive movie/TV show review
- **2** — Negative movie/TV show review

**Approach:** TF-IDF feature extraction (word + character n-grams) with Logistic Regression classifier, selected through systematic model comparison and hyperparameter tuning.

**Dataset:** Based on Pang & Lee (2004) sentiment data, supplemented with additional sources.


## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')

from scipy.sparse import hstack
from sklearn.model_selection import StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

print("All imports successful.")

## 2. Load Data

In [ ]:
train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')
sample_sub = pd.read_csv('data/sample_submission.csv')

print(f"Train: {train.shape}")
print(f"Test:  {test.shape}")
print(f"Sample submission: {sample_sub.shape}")

## 3. Exploratory Data Analysis

Examining class balance, text length distributions, and data quality.

In [ ]:
print("=== Class Distribution ===")
print(train['LABEL'].value_counts().sort_index())
print()
print("=== Class Proportions ===")
print(train['LABEL'].value_counts(normalize=True).sort_index().round(4))

In [ ]:
train['text_len'] = train['TEXT'].astype(str).apply(len)
train['word_count'] = train['TEXT'].astype(str).apply(lambda x: len(x.split()))

print("=== Text Length (characters) by Label ===")
print(train.groupby('LABEL')['text_len'].describe().round(1))
print()
print("=== Word Count by Label ===")
print(train.groupby('LABEL')['word_count'].describe().round(1))
print(f"\nNull TEXT values: {train['TEXT'].isnull().sum()}")
print(f"Empty TEXT values: {(train['TEXT'].astype(str).str.strip() == '').sum()}")

**Observations:**
- Class 0 (non-reviews) makes up ~46% of the data with significantly shorter texts (mean ~77 words vs ~210 for reviews)
- Classes 1 and 2 (positive/negative reviews) are nearly balanced and have similar length distributions
- 7 null and 5 empty text values need to be handled during preprocessing
- The primary classification challenge is distinguishing positive from negative reviews, since non-reviews differ substantially in vocabulary and length

## 4. Text Preprocessing

The raw text contains HTML tags (from web scraping), URLs, and inconsistent whitespace. We apply the following cleaning steps:
1. Remove `<br />` and other HTML tags
2. Remove URLs
3. Collapse whitespace
4. Convert to lowercase

In [ ]:
def clean_text(text):
    """Clean raw text for classification."""
    if pd.isna(text):
        return ""
    text = str(text)
    text = re.sub(r'<br\s*/?>', ' ', text)   # HTML line breaks
    text = re.sub(r'<[^>]+>', ' ', text)       # other HTML tags
    text = re.sub(r'http\S+|www\.\S+', ' ', text)  # URLs
    text = re.sub(r'\s+', ' ', text).strip()  # normalize whitespace
    text = text.lower()
    return text

train['clean_text'] = train['TEXT'].apply(clean_text)
test['clean_text'] = test['TEXT'].apply(clean_text)

print("Cleaning complete.")
print(f"Sample (label 0): {train[train['LABEL']==0]['clean_text'].iloc[0][:120]}...")
print(f"Sample (label 1): {train[train['LABEL']==1]['clean_text'].iloc[0][:120]}...")
print(f"Sample (label 2): {train[train['LABEL']==2]['clean_text'].iloc[0][:120]}...")

## 5. Feature Extraction: TF-IDF

We construct two complementary TF-IDF representations and combine them:

**Word-level TF-IDF (unigrams + bigrams):**
- Captures individual words and two-word phrases (e.g., "worst movie", "highly recommend")
- 100,000 features with `sublinear_tf=True` (log-scaled term frequency)

**Character-level TF-IDF (2 to 5 character windows):**
- Captures sub-word patterns and is robust to misspellings
- Provides signal for garbled/non-English text (common in class 0)
- 50,000 features using `char_wb` analyzer (respects word boundaries)

Both use `min_df=2` (ignore terms appearing only once) and `max_df=0.95` (ignore terms in >95% of documents).

In [ ]:
tfidf_word = TfidfVectorizer(
    max_features=100000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True,
    strip_accents='unicode',
    analyzer='word',
    token_pattern=r'\w{1,}',
)

tfidf_char = TfidfVectorizer(
    max_features=50000,
    ngram_range=(2, 5),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True,
    strip_accents='unicode',
    analyzer='char_wb',
)

X_train_word = tfidf_word.fit_transform(train['clean_text'])
X_test_word = tfidf_word.transform(test['clean_text'])

X_train_char = tfidf_char.fit_transform(train['clean_text'])
X_test_char = tfidf_char.transform(test['clean_text'])

X_train = hstack([X_train_word, X_train_char])
X_test = hstack([X_test_word, X_test_char])
y_train = train['LABEL']

print(f"Word features:  {X_train_word.shape[1]:,}")
print(f"Char features:  {X_train_char.shape[1]:,}")
print(f"Combined train: {X_train.shape}")
print(f"Combined test:  {X_test.shape}")

## 6. Model Comparison

We evaluate three classifiers commonly used for text classification using 5-fold stratified cross-validation:

1. **Logistic Regression** — multinomial with L2 regularization; standard baseline for text
2. **LinearSVC** — support vector classifier with linear kernel; maximizes the margin between classes
3. **Multinomial Naive Bayes** — probabilistic classifier based on word frequency distributions

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Logistic Regression
lr = LogisticRegression(C=1.0, max_iter=1000, solver='lbfgs',
                        multi_class='multinomial', random_state=42)
lr_scores = cross_val_score(lr, X_train, y_train, cv=cv, scoring='accuracy', n_jobs=-1)
print(f"Logistic Regression:  {lr_scores.mean():.4f} (+/- {lr_scores.std():.4f})")

# LinearSVC
svc = LinearSVC(C=1.0, max_iter=2000, random_state=42)
svc_scores = cross_val_score(svc, X_train, y_train, cv=cv, scoring='accuracy', n_jobs=-1)
print(f"LinearSVC:            {svc_scores.mean():.4f} (+/- {svc_scores.std():.4f})")

# Multinomial Naive Bayes
mnb = MultinomialNB(alpha=0.1)
nb_scores = cross_val_score(mnb, X_train, y_train, cv=cv, scoring='accuracy', n_jobs=-1)
print(f"Multinomial NB:       {nb_scores.mean():.4f} (+/- {nb_scores.std():.4f})")

**Result:** Logistic Regression and LinearSVC perform comparably (~93.3%), both substantially outperforming Multinomial Naive Bayes (~87.8%). We select Logistic Regression for hyperparameter tuning since it offers probability estimates natively and is well-suited for multinomial classification.

## 7. Hyperparameter Tuning

Tuning the regularization parameter `C` for Logistic Regression via grid search with 5-fold CV. Lower C means stronger regularization; higher C allows the model to fit training data more closely.

In [ ]:
param_grid = {'C': [0.1, 0.5, 1.0, 2.0, 5.0, 10.0]}

grid = GridSearchCV(
    LogisticRegression(max_iter=1000, solver='lbfgs',
                       multi_class='multinomial', random_state=42),
    param_grid,
    cv=cv,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)
grid.fit(X_train, y_train)

print(f"\nBest C: {grid.best_params_['C']}")
print(f"Best CV Accuracy: {grid.best_score_:.4f}")

# Show all results
results = pd.DataFrame(grid.cv_results_)[['param_C', 'mean_test_score', 'std_test_score']]
results.columns = ['C', 'Mean Accuracy', 'Std']
print(f"\n{results.to_string(index=False)}")

## 8. Error Analysis

Examining where the tuned model makes mistakes on the training data to understand the classification boundaries.

In [ ]:
best_model = grid.best_estimator_
train_preds = best_model.predict(X_train)

print(f"Training errors: {(train_preds != y_train).sum()} / {len(y_train)}")
print(f"Training accuracy: {accuracy_score(y_train, train_preds):.4f}")
print(f"\nConfusion Matrix:")
print(confusion_matrix(y_train, train_preds))
print()
print(classification_report(y_train, train_preds,
                            target_names=['Not Review (0)', 'Positive (1)', 'Negative (2)']))

In [ ]:
# Show example misclassifications
misclassified = train[train_preds != y_train].copy()
misclassified['predicted'] = train_preds[train_preds != y_train]

for true_label in [0, 1, 2]:
    for pred_label in [0, 1, 2]:
        if true_label != pred_label:
            subset = misclassified[(misclassified['LABEL'] == true_label) &
                                   (misclassified['predicted'] == pred_label)]
            if len(subset) > 0:
                print(f"\n--- True={true_label}, Predicted={pred_label} ({len(subset)} cases) ---")
                print(subset['clean_text'].iloc[0][:200])

**Error Analysis Findings:**
- Class 0 is classified with near-perfect precision and recall (~99%)
- The main confusion is between positive (1) and negative (2) reviews — these share similar vocabulary and sentence structures, differing primarily in sentiment polarity
- Common misclassifications include: book/game reviews wrongly classified as movie reviews, short or ambiguous texts, and reviews with mixed sentiment

## 9. Generate Submission

Training the tuned Logistic Regression (best C from grid search) on the full training set and generating predictions for the test data.

In [ ]:
# Train on full data with best hyperparameters
final_model = grid.best_estimator_
test_preds = final_model.predict(X_test)

# Create submission
submission = pd.DataFrame({
    'ID': test['ID'],
    'LABEL': test_preds
})

# Sanity checks
print("=== Submission Sanity Checks ===")
print(f"Shape: {submission.shape} (expected {sample_sub.shape})")
print(f"Label distribution:")
print(submission['LABEL'].value_counts().sort_index())
print(f"\nAny nulls: {submission.isnull().any().any()}")
print(f"ID dtype:    {submission['ID'].dtype}")
print(f"LABEL dtype: {submission['LABEL'].dtype}")

# Save
submission.to_csv('data/submission.csv', index=False)
print("\nSubmission saved to data/submission.csv")

## 10. Summary

### Pipeline
1. **Preprocessing:** Removed HTML tags, URLs, normalized whitespace, lowercased
2. **Features:** TF-IDF with word bigrams (100K features) and character n-grams (50K features), combined into a 150K-dimensional sparse feature matrix
3. **Model Selection:** Compared Logistic Regression, LinearSVC, and Multinomial Naive Bayes via 5-fold stratified CV
4. **Tuning:** Grid search over regularization strength C for Logistic Regression
5. **Final Model:** Multinomial Logistic Regression with L2 regularization (C tuned via CV)

### Key Design Decisions
- **TF-IDF over Bag-of-Words:** Sublinear TF-IDF downweights very common terms and upweights discriminative ones, improving classification accuracy
- **Character n-grams:** Added robustness to misspellings and captured sub-word patterns; also helped identify garbled/non-English text in class 0
- **Logistic Regression:** Chosen over LinearSVC (comparable accuracy) because it provides calibrated probability estimates and is interpretable via feature weights
- **Multinomial NB rejected:** ~5.5% lower accuracy due to its strong feature independence assumption, which is violated with n-gram features

### Results
- **Cross-validated accuracy:** ~93.5% (5-fold stratified)
- **Kaggle leaderboard accuracy:** reported upon submission
